<a href="https://colab.research.google.com/github/murilo-guimaraes/sistemas-operacionais/blob/main/Mem%C3%B3ria_Virtual_e_Pagina%C3%A7%C3%A3o.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
#**MEMÓRIA VIRTUAL E PAGINAÇÃO**

####**Relatório Técnico Acadêmico | Sistemas Operacionais 2026 | Colaboratory**
**Data: 12/05/2026**
<div>Memória Virtual | Paginação | MMU | Algoritmos de Substituição | Swapping</div>

*Baseado na Unidade 4 Seção 3 do livro **Sistemas Operacionais** (Cynthia da Silva Barbosa)*

##**Introdução Teórica**

A Memória Virtual é uma técnica sofisticada que permite a execução de processos cujo tamanho excede a capacidade da memória física (RAM) disponível. Segundo Barbosa (2018), esse conceito isola a memória lógica da memória física, utilizando a unidade de gerenciamento de memória (MMU) para mapear endereços virtuais em molduras reais. Através da paginação e de algoritmos estratégicos de substituição, o sistema operacional garante que as partes ativas de um programa permaneçam na RAM, enquanto o restante é mantido em armazenamento secundário, otimizando a multiprogramação e a eficiência do hardware.

---
##**Objetivos**

* Compreender o funcionamento da Memória Virtual e o papel da MMU.
* Analisar o processo de Paginação e o fenômeno de Falta de Página (*Page Fault*).
* Simular e visualizar a eficiência dos algoritmos de substituição de páginas (FIFO e LRU).
* Diferenciar as abordagens de Paginação e Segmentação na gestão de memória.

---
###**1. Simulador de MMU: Tradução de Endereços**

Neste teste, criamos um simulador funcional da Unidade de Gerenciamento de Memória (MMU). O código recebe um endereço virtual, consulta uma tabela de páginas real e verifica se a página está na RAM ou se deve gerar uma interrupção de *Page Fault*. Conforme Barbosa, este é o coração da memória virtual: a abstração do endereçamento.

In [6]:
import random

class MMU:
    def __init__(self, num_paginas):
        # Tabela de Páginas: {Página Virtual: [Moldura Física, Bit Presente]}
        self.tabela_paginas = {i: [random.randint(0, 50), random.choice([0, 1])] for i in range(num_paginas)}
        self.tamanho_pagina = 4096 # 4KB

    def traduzir(self, endereco_virtual):
        pagina_v = endereco_virtual // self.tamanho_pagina
        offset = endereco_virtual % self.tamanho_pagina

        print(f"[*] Tentando acessar endereço virtual: {endereco_virtual}")

        if self.tabela_paginas[pagina_v][1] == 1: # Bit presente == 1
            moldura = self.tabela_paginas[pagina_v][0]
            endereco_fisico = (moldura * self.tamanho_pagina) + offset
            return f"[SUCESSO] Página {pagina_v} na RAM. Endereço Físico: {endereco_fisico}"
        else:
            return f"[PAGE FAULT] Página {pagina_v} não está na RAM. SO deve buscar no Disco."

# Execução Prática
mmu_simulada = MMU(10)
print(mmu_simulada.traduzir(12500)) # Exemplo de acesso
print(mmu_simulada.traduzir(40000)) # Exemplo de acesso

[*] Tentando acessar endereço virtual: 12500
[PAGE FAULT] Página 3 não está na RAM. SO deve buscar no Disco.
[*] Tentando acessar endereço virtual: 40000
[PAGE FAULT] Página 9 não está na RAM. SO deve buscar no Disco.


####**O que observar:** <a>O simulador demonstra a lógica binária de decisão do hardware. Quando o bit de presença é 0, o código interrompe a tradução e sinaliza o Page Fault. Como destaca Barbosa, essa verificação é feita a cada instrução do processador, exigindo que o mapeamento seja extremamente veloz para não comprometer a performance do sistema.</a>

---
###**2. Algoritmo LRU (Least Recently Used) na Prática**

Diferente do FIFO, o LRU é um algoritmo inteligente que remove a página que ficou mais tempo sem ser acessada. Este código simula a "fila de prioridade" da memória RAM, mostrando exatamente qual página o SO escolhe para "expulsar" quando um novo dado precisa entrar e o espaço está cheio.

In [7]:
from collections import deque

def simulador_lru(capacidade, sequencia):
    memoria = []
    ordem_uso = deque() # Controla quem foi usado por último

    print(f"Iniciando RAM com capacidade para {capacidade} quadros...\n")

    for pag in sequencia:
        if pag not in memoria:
            if len(memoria) >= capacidade:
                # Remove a página que está no início da fila (menos usada recentemente)
                removida = ordem_uso.popleft()
                memoria.remove(removida)
                print(f"[SUBSTITUIÇÃO] RAM cheia! Removendo página {removida} (LRU)")

            memoria.append(pag)
            ordem_uso.append(pag)
            print(f" -> Página {pag} carregada na RAM: {memoria}")
        else:
            # "Refresh" na página: move para o final da fila de uso
            ordem_uso.remove(pag)
            ordem_uso.append(pag)
            print(f" -> Página {pag} já estava na RAM. Status atual: {memoria}")

# Sequência de teste do livro
sequencia_paginas = [1, 2, 3, 2, 1, 4, 5]
simulador_lru(3, sequencia_paginas)

Iniciando RAM com capacidade para 3 quadros...

 -> Página 1 carregada na RAM: [1]
 -> Página 2 carregada na RAM: [1, 2]
 -> Página 3 carregada na RAM: [1, 2, 3]
 -> Página 2 já estava na RAM. Status atual: [1, 2, 3]
 -> Página 1 já estava na RAM. Status atual: [1, 2, 3]
[SUBSTITUIÇÃO] RAM cheia! Removendo página 3 (LRU)
 -> Página 4 carregada na RAM: [1, 2, 4]
[SUBSTITUIÇÃO] RAM cheia! Removendo página 2 (LRU)
 -> Página 5 carregada na RAM: [1, 4, 5]


####**O que observar:** <a>Note como a página 3 foi a escolhida para sair quando a página 4 entrou. Isso ocorreu porque as páginas 1 e 2 foram acessadas mais recentemente (sofreram "refresh"). Conforme Cynthia Barbosa descreve, o LRU é muito mais eficiente que o FIFO pois preserva as páginas que o programa está utilizando ativamente no momento.</a>

---
###**3. Gerência de Swapping: RAM vs. Disco**

Este teste simula o movimento de processos entre a Memória Principal e a Memória Secundária (Área de Swap). O código monitora o uso da RAM e, ao atingir o limite crítico, "desce" processos inativos para o disco rígido para liberar espaço para novas execuções, demonstrando a técnica de overlay automático.

In [10]:
class MemoriaSistema:
    def __init__(self, limite_ram):
        self.ram = {}
        self.disco = {}
        self.limite_ram = limite_ram

    def carregar_processo(self, nome, tamanho):
        uso_atual = sum(self.ram.values())

        if uso_atual + tamanho > self.limite_ram:
            print(f"[ALERTA] RAM Insuficiente para {nome} ({tamanho}MB). Iniciando SWAP...")
            # Move o primeiro processo da RAM para o Disco (Swap Out)
            p_nome = list(self.ram.keys())[0]
            p_tam = self.ram.pop(p_nome)
            self.disco[p_nome] = p_tam
            print(f" << SWAP OUT: Processo {p_nome} movido para o DISCO.")

        self.ram[nome] = tamanho
        print(f" >> CARREGADO: Processo {nome} está na RAM.")
        print(f" ESTADO ATUAL -> RAM: {list(self.ram.keys())} | DISCO: {list(self.disco.keys())}\n")

# Simulação
os_mem = MemoriaSistema(limite_ram=100) # RAM de 100MB
os_mem.carregar_processo("Navegador", 40)
os_mem.carregar_processo("Jogo", 50)
os_mem.carregar_processo("Gravador", 30) # Vai estourar os 100MB

 >> CARREGADO: Processo Navegador está na RAM.
 ESTADO ATUAL -> RAM: ['Navegador'] | DISCO: []

 >> CARREGADO: Processo Jogo está na RAM.
 ESTADO ATUAL -> RAM: ['Navegador', 'Jogo'] | DISCO: []

[ALERTA] RAM Insuficiente para Gravador (30MB). Iniciando SWAP...
 << SWAP OUT: Processo Navegador movido para o DISCO.
 >> CARREGADO: Processo Gravador está na RAM.
 ESTADO ATUAL -> RAM: ['Jogo', 'Gravador'] | DISCO: ['Navegador']



####**O que observar:** <a>O código demonstra o "atrito" entre a necessidade de execução e a limitação física. O processo "Navegador" foi movido para o disco para que o "Gravador" pudesse rodar. Segundo Barbosa, o swap permite a ilusão de memória infinita, mas ao custo de performance, já que o disco é ordens de magnitude mais lento que a RAM.</a>

---
##**Conclusão**

A análise prática da Memória Virtual e Paginação confirmou que essas técnicas são vitais para a estabilidade dos sistemas modernos. O uso da MMU e a aplicação de algoritmos eficientes como o NUR/LRU permitem que o hardware seja aproveitado ao máximo, mascarando a latência do disco rígido. Conforme concluído por Cynthia Barbosa, a compreensão desse mapeamento entre endereços virtuais e físicos é o que diferencia um desenvolvedor de ADS na criação de aplicações que gerenciam recursos de forma resiliente e segura.

---
##**Referências Bibliográficas**

BARBOSA, Cynthia da Silva. **Sistemas operacionais**: Memória Virtual. Londrina: Editora e Distribuidora Educacional S.A., 2018.

TANENBAUM, Andrew S. **Sistemas Operacionais Modernos**. 4. ed. São Paulo: Pearson, 2016.

---
<p align="center"><b>© 2026 Murilo Guimarães. Acadêmico de ADS.</b></p>

---